# Eliminating outlier retention with multi-pass GF4 + *selective* weight residual
### Llama-3-8B + OPT-6.7B

**Setting = our outlier-retention results:** randomized-Hadamard rotation, **NVFP4 weights** (E2M1 +
per-16 E4M3 block scale, RTN — *no* Hessian), **GF4 activations**. W4A4 on all transformer linears;
`lm_head`/embeddings FP16. Only the MLP down-projection (`down_proj` Llama / `fc2` OPT) changes.

**Why selective.** On Llama-2-7B, multi-passing only the down-proj *activation* plateaued ~0.12 PPL above
FP16 retention — the leftover was the down-proj *weight's* 1-pass NVFP4 error. But retention never holds
a whole layer; it holds the **outlier channels**. Since we Hadamard-rotate (which smears native channel
outliers), we define the outlier set in the **rotated domain**: the down-proj output **rows with the
largest first-pass NVFP4 error**. Only those rows get a 2nd residual weight pass.

**Conditions** (activation fixed at 2-pass residual GF4 for the weight sweep, so the weight is isolated):

| condition | down-proj weight | down-proj activation |
|---|---|---|
| retention (baseline) | FP16 | FP16 |
| 1-pass (plain W4A4) | NVFP4 ×1 | GF4 ×1 |
| act-only N-pass | NVFP4 ×1 | GF4 ×N |
| **selective weight**, top-k% rows | NVFP4 ×1, **+2nd pass on top-k% rows** | GF4 ×2 |

We also print the **error-concentration curve** (what fraction of first-pass weight error the top-k% rows
hold) and the **effective bits/weight** on the down-projection, so you see how few channels are needed.

> Runs Llama-3-8B then OPT-6.7B. A100-40GB. Llama-3 gated — set `HF_TOKEN` (Colab Secrets).

In [ ]:
!pip -q install "transformers>=4.44" datasets accelerate sentencepiece

In [ ]:
import os, gc, math, torch, torch.nn as nn, torch.nn.functional as F

MODELS   = [
    "facebook/opt-125m",              # proper NSAMP=40 (trivial fit)
    "facebook/opt-13b",               # now fits 40GB with the wrap_model free-as-you-go fix
    "meta-llama/Llama-2-13b-hf",      # fits 40GB with the fix
    "facebook/opt-30b",               # STILL needs A100-80GB (60GB fp16); will OOM on 40GB (caught, skipped)
]   # remaining-only rerun; 1.3b/2.7b/6.7b/Llama-3-8B already banked in writeups/retention_suite_results.md
SEQLEN   = 2048
NSAMP    = 40
BLOCK    = 16
CLIP     = 2.5
ACT_LADDER = (2, 4)               # activation-only residual passes to show saturation
KSEL     = (0.05, 0.10, 0.25, 0.50, 1.00)   # top-k% rows getting the 2nd weight pass
ACT_FIX  = 2                      # activation passes held fixed during the weight sweep
DEV      = "cuda"

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass
HF_TOKEN = os.environ.get("HF_TOKEN")

DOWN_SUBSTR = ("down_proj", "fc2", "dense_4h_to_h")
KEEP_FP16   = ("lm_head", "embed_out", "embed_tokens", "embed_positions")

In [ ]:
# ---- codebooks + kernels --------------------------------------------------
GF4_POS = torch.tensor([0.0, 0.0796082, 0.1737177, 0.2828685,
                        0.3952704, 0.5250730, 0.6961928, 1.0])
E2M1    = torch.tensor([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])

def fwht(x):
    """Normalized fast Walsh-Hadamard transform along last dim (n = power of 2)."""
    n = x.shape[-1]; orig = x.shape
    x = x.reshape(-1, n).clone()
    h = 1
    while h < n:
        x = x.view(x.shape[0], n // (2 * h), 2, h)
        a = x[:, :, 0, :].clone(); b = x[:, :, 1, :].clone()
        x[:, :, 0, :] = a + b
        x[:, :, 1, :] = a - b
        x = x.reshape(x.shape[0], n)
        h *= 2
    return (x / math.sqrt(n)).reshape(orig)

def rotate(x, D, Kpad):
    """Randomized Hadamard (pad -> +-1 signs D -> FWHT); same D folded into weight => exact."""
    K = x.shape[-1]
    if Kpad != K: x = F.pad(x, (0, Kpad - K))
    return fwht(x * D)

def to_e4m3(x):
    x = x.float().clamp(min=2.0 ** -9, max=448.0)
    e = torch.floor(torch.log2(x)).clamp(min=-6.0); s = 2.0 ** e
    return (torch.round((x / s) * 8) / 8 * s).clamp(max=448.0)

def quant_nvfp4(W, block=BLOCK):
    """NVFP4 weight quant (RTN): per-16 E4M3 block scale + nearest E2M1."""
    shape = W.shape
    x = W.reshape(-1, block).float()
    absmax = x.abs().amax(-1, keepdim=True).clamp(min=1e-8)
    scale  = to_e4m3(absmax / 6.0)
    xn = x / scale; sign = xn.sign(); mag = xn.abs().clamp(max=6.0)
    q = E2M1.to(x.device)
    idx = (mag.unsqueeze(-1) - q.view(1, 1, -1)).abs().argmin(-1)
    return (sign * q[idx] * scale).reshape(shape).to(W.dtype)

def gf4(x, block=BLOCK, clip=CLIP):
    shape = x.shape; K = shape[-1]
    x2 = x.reshape(-1, K).float()
    pad = (block - K % block) % block
    xp = F.pad(x2, (0, pad)); Kp = xp.shape[1]
    xb = xp.reshape(-1, block)
    rms = xb.pow(2).mean(-1, keepdim=True).sqrt().clamp(min=1e-8)
    scale = rms * clip
    sign = xb.sign(); xn = (xb.abs() / scale).clamp(0, 1)
    L = GF4_POS.to(x.device)
    q = L[(xn.unsqueeze(-1) - L.view(1, 1, -1)).abs().argmin(-1)]
    xh = (sign * scale * q).reshape(x2.shape[0], Kp)[:, :K]
    return xh.reshape(shape).to(x.dtype)

def gf4_npass(x, n, block=BLOCK, clip=CLIP):
    xf = x.float(); q = torch.zeros_like(xf)
    for _ in range(n): q = q + gf4(xf - q, block, clip)
    return q.to(x.dtype)

In [ ]:
# ---- quantized linears ----------------------------------------------------
def _sign_vec(Kpad):
    g = torch.Generator().manual_seed(Kpad)
    return (torch.randint(0, 2, (Kpad,), generator=g).float() * 2 - 1)

class QuantLinear(nn.Module):
    """Bulk W4A4: rotated NVFP4 weight + rotated 1-pass GF4 activation."""
    def __init__(self, lin, block=BLOCK, clip=CLIP):
        super().__init__()
        W = lin.weight.data.float().cpu(); out, K = W.shape
        Kpad = 1 << (K - 1).bit_length(); D = _sign_vec(Kpad)
        self.register_buffer("D",  D.half())
        self.register_buffer("Wq", quant_nvfp4(rotate(W, D, Kpad), block).half())
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.data.clone(), requires_grad=False)
        self.Kpad, self.block, self.clip = Kpad, block, clip
    def forward(self, x):
        xr = rotate(x.float(), self.D.float().to(x.device), self.Kpad)
        y  = gf4(xr, self.block, self.clip).half() @ self.Wq.to(x.device).t()
        if self.bias is not None: y = y + self.bias.to(y.dtype)
        return y.to(x.dtype)

class SwitchDownProj(nn.Module):
    """The down-projection under test. prepare() sets the condition:
      mode='fp16'  -> retention (weight+act FP16)
      mode='quant' -> activation = apasses-pass GF4;
                      weight = 1-pass NVFP4, PLUS a 2nd residual pass on the top-`ksel`
                      fraction of OUTPUT ROWS ranked by first-pass rotated NVFP4 error
                      (ksel=1.0 -> whole layer; ksel=0 / apasses-only -> plain 1-pass weight).
    Keeps only W_fp16 resident; the quantized weight is (re)built in prepare()."""
    def __init__(self, lin, block=BLOCK, clip=CLIP):
        super().__init__()
        W = lin.weight.data; out, K = W.shape
        self.Kpad = 1 << (K - 1).bit_length()
        self.register_buffer("D",      _sign_vec(self.Kpad).half())
        self.register_buffer("W_fp16", W.data.clone().half())
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.data.clone(), requires_grad=False)
        self.block, self.clip = block, clip
        self.mode, self.apasses, self.ksel_eff, self._Wq = "fp16", 1, 0.0, None

    @torch.no_grad()
    def row_error(self):
        """Per output-row first-pass NVFP4 error norm in the rotated domain."""
        Wr = rotate(self.W_fp16.float(), self.D.float(), self.Kpad)
        return (Wr - quant_nvfp4(Wr, self.block)).float().norm(dim=1)

    @torch.no_grad()
    def prepare(self, mode, apasses=1, ksel=0.0):
        self.mode, self.apasses = mode, apasses
        if mode == "fp16":
            self._Wq = None; self.ksel_eff = 0.0; return
        Wr  = rotate(self.W_fp16.float(), self.D.float(), self.Kpad)
        Wq1 = quant_nvfp4(Wr, self.block)                       # 1-pass weight everywhere
        if ksel and ksel > 0:                                   # 2nd residual pass on top-k% rows
            out  = Wr.shape[0]
            ntop = max(1, int(round(ksel * out)))
            idx  = torch.topk((Wr - Wq1).float().norm(dim=1), ntop).indices
            Wq1[idx] = Wq1[idx] + quant_nvfp4(Wr[idx] - Wq1[idx], self.block)
            self.ksel_eff = ntop / out
        else:
            self.ksel_eff = 0.0
        self._Wq = Wq1.half(); del Wr

    def forward(self, x):
        if self.mode == "fp16":
            y = x.to(self.W_fp16.dtype) @ self.W_fp16.t()
        else:
            xr = rotate(x.float(), self.D.float().to(x.device), self.Kpad)
            y  = gf4_npass(xr, self.apasses, self.block, self.clip).half() @ self._Wq.to(x.device).t()
        if self.bias is not None: y = y + self.bias.to(y.dtype)
        return y.to(x.dtype)

In [ ]:
# ---- surgery + eval -------------------------------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

def _set(model, name, mod):
    parent = model.get_submodule(name.rsplit(".", 1)[0]) if "." in name else model
    setattr(parent, name.rsplit(".", 1)[-1], mod)

def wrap_model(model):
    downs = []
    for name, mod in list(model.named_modules()):
        if not isinstance(mod, nn.Linear): continue
        if any(k in name for k in KEEP_FP16): continue
        if any(s in name for s in DOWN_SUBSTR):
            sw = SwitchDownProj(mod).to(DEV); _set(model, name, sw); downs.append(sw)
        else:
            _set(model, name, QuantLinear(mod).to(DEV))
        # free the ORIGINAL fp16 weight now: list(named_modules()) above pins `mod`,
        # so without this the fp16 weight and its quantized Wq copy coexist (~2x mem =>
        # 13B OOMs on a 40GB A100). QuantLinear/SwitchDownProj already copied what they need.
        mod.weight = None
        if getattr(mod, "bias", None) is not None: mod.bias = None
    torch.cuda.empty_cache()
    return downs

def set_down(downs, mode, apasses=1, ksel=0.0):
    for d in downs: d.prepare(mode, apasses, ksel)

@torch.no_grad()
def eval_ppl(model, enc):
    n = enc.numel() // SEQLEN; nll = cnt = 0
    for i in range(min(n, NSAMP)):
        ids = enc[:, i*SEQLEN:(i+1)*SEQLEN].to(DEV)
        nll += model(ids, labels=ids, use_cache=False).loss.float().item() * (SEQLEN - 1)
        del ids; torch.cuda.empty_cache()   # KV cache off + free per-window: fits 13B eval on 40GB
        cnt += SEQLEN - 1
    return math.exp(nll / cnt)

@torch.no_grad()
def concentration(downs, ks):
    """Fraction of total first-pass down-proj weight error held by the top-k% rows (avg / layer)."""
    acc = {k: 0.0 for k in ks}
    for d in downs:
        e, _ = torch.sort(d.row_error(), descending=True); tot = e.sum().clamp(min=1e-9)
        for k in ks:
            n = max(1, int(round(k * len(e)))); acc[k] += (e[:n].sum() / tot).item()
    return {k: acc[k] / len(downs) for k in ks}

In [ ]:
# ---- run both models ------------------------------------------------------
def wbits(ksel_eff):                      # naive effective bits/weight on down-proj (ignore scale)
    return 4.0 * (1.0 + ksel_eff)         # top-ksel rows at 8b (2 passes), rest at 4b

def run(model_id):
    print(f"\n{'='*74}\n== {model_id}: eliminate down-proj retention (WikiText-2) ==\n{'='*74}")
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=False, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16,
                                                 low_cpu_mem_usage=True,
                                                 token=HF_TOKEN).to(DEV).eval()
    test = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    enc  = tok("\n\n".join(test["text"]), return_tensors="pt").input_ids
    downs = wrap_model(model)
    print(f"  {len(downs)} down-proj layers switchable; rest bulk W4A4")

    conc = concentration(downs, KSEL)
    print("  error concentration (top-k% rows hold X% of 1st-pass weight error): "
          + ", ".join(f"{int(k*100)}%->{v*100:.0f}%" for k, v in conc.items()))
    print(f"  {'down-proj treatment':38s} {'PPL':>8s} {'dPPL':>8s} {'~Wbits':>7s}")

    set_down(downs, "fp16"); base = eval_ppl(model, enc)
    print(f"  {'FP16 retention (baseline)':38s} {base:8.3f} {'--':>8s} {'16':>7s}")

    set_down(downs, "quant", 1, 0.0); v = eval_ppl(model, enc)
    print(f"  {'1-pass FP4 (plain W4A4)':38s} {v:8.3f} {v-base:+8.3f} {'4.0':>7s}")

    for a in ACT_LADDER:                                   # activation-only ladder (weight 1-pass)
        set_down(downs, "quant", a, 0.0); v = eval_ppl(model, enc)
        print(f"  {f'act-only {a}-pass (W 1-pass)':38s} {v:8.3f} {v-base:+8.3f} {'4.0':>7s}")

    for k in KSEL:                                         # selective weight residual (act fixed)
        set_down(downs, "quant", ACT_FIX, k); v = eval_ppl(model, enc)
        eff = downs[0].ksel_eff
        tag = f"act{ACT_FIX} + W 2-pass top-{int(k*100)}% rows"
        print(f"  {tag:38s} {v:8.3f} {v-base:+8.3f} {wbits(eff):7.2f}")

    print("  -> smallest (k, Wbits) whose row ~ baseline = retention eliminated; compare to the\n"
          "     concentration curve: if top-10% rows hold most error, ~4.4 Wbits should suffice.")
    del model, downs; gc.collect(); torch.cuda.empty_cache()

for m in MODELS:
    try:
        run(m)
    except Exception as e:
        print(f"  [{m}] FAILED: {type(e).__name__}: {str(e)[:120]}")
        gc.collect(); torch.cuda.empty_cache()

## Reading the result

- **error concentration** line first: if the top ~10% of rows hold most of the first-pass weight error,
  the premise holds — a selective 2nd pass on those rows recovers nearly all the gain. If it's flat
  (top-10% ≈ 10%), the rotation already equalized the weight and selective residual won't help much.
- **1-pass** = plain W4A4 (no retention) → the collapse. **act-only** shows activation saturation (and
  the residual weight floor it plateaus at). **selective weight** rows: find the smallest `k` / `~Wbits`
  whose ΔPPL ≈ 0. That `k` is how few down-proj channels need the extra pass to replace FP16 retention.
- Cost: `~Wbits` is the down-proj weight cost; e.g. top-10% rows at 2 passes ≈ **4.4 bits** vs the 16-bit
  retained layer — the efficiency argument, on the *existing* FP4 array, no dedicated FP16 datapath.